In [ ]:

import time


#NAIVE STRING MATCHING

def naive_search(text, pattern):

    #Slide the pattern across the text one character at a time and compare.

    n, m = len(text), len(pattern)
    matches = []

    if m == 0 or m > n:
        return matches

    for i in range(n - m + 1):
        j = 0
        while j < m and text[i + j] == pattern[j]:
            j += 1
        if j == m:
            matches.append(i)

    return matches

#KNUTH-MORRIS-PRATT (KMP)

def _build_lps(pattern):

    #Build the "Longest Proper Prefix which is also a Suffix" (LPS) array.

    m = len(pattern)
    lps = [0] * m
    length = 0   # length of the previous longest prefix-suffix
    i = 1

    while i < m:
        if pattern[i] == pattern[length]:
            length += 1
            lps[i] = length
            i += 1
        elif length != 0:
            length = lps[length - 1]   # fall back, don't advance i
        else:
            lps[i] = 0
            i += 1

    return lps


def kmp_search(text, pattern):

    #Search using the Knuth-Morris-Pratt algorithm.


    n, m = len(text), len(pattern)
    matches = []

    if m == 0 or m > n:
        return matches

    lps = _build_lps(pattern)
    i = j = 0  # i -> index in text, j -> index in pattern

    while i < n:
        if text[i] == pattern[j]:
            i += 1
            j += 1
            if j == m:
                matches.append(i - j)
                j = lps[j - 1]
        elif j != 0:
            j = lps[j - 1]
        else:
            i += 1

    return matches


# BOYER-MOORE (Bad Character Heuristic)

def _build_bad_char_table(pattern):

    #For each character in pattern, record the index of its last occurrence.

    table = {}
    for i, ch in enumerate(pattern):
        table[ch] = i
    return table


def boyer_moore_search(text, pattern):

   # Search using the Boyer-Moore algorithm with the Bad Character heuristic.


    n, m = len(text), len(pattern)
    matches = []

    if m == 0 or m > n:
        return matches

    bad_char = _build_bad_char_table(pattern)
    shift = 0  # shift of the pattern relative to text

    while shift <= n - m:
        j = m - 1
        # Compare from the rightmost character of the pattern backwards
        while j >= 0 and pattern[j] == text[shift + j]:
            j -= 1

        if j < 0:
            # Full match found
            matches.append(shift)
            # Shift enough to align the next occurrence of the last
            # matched character (or by 1 if pattern exhausted / char absent)
            next_char = text[shift + m] if shift + m < n else None
            shift += (m - bad_char.get(next_char, -1)) if next_char is not None else 1
        else:
            mismatched_char = text[shift + j]
            last_occurrence = bad_char.get(mismatched_char, -1)
            shift += max(1, j - last_occurrence)

    return matches


#DEMO / TEST HARNESS
def time_it(func, text, pattern):
    start = time.perf_counter()
    result = func(text, pattern)
    elapsed = (time.perf_counter() - start) * 1000  # ms
    return result, elapsed


def main():
    text = (
        "ABABDABACDABABCABAB the quick brown fox jumps over the lazy dog "
        "ABABCABAB and again ABABDABACDABABCABAB for good measure"
    )
    pattern = "ABABCABAB"

    print(f"Text length   : {len(text)} characters")
    print(f"Pattern       : '{pattern}' (length {len(pattern)})\n")

    algorithms = [
        ("Naive Search", naive_search),
        ("Knuth-Morris-Pratt (KMP)", kmp_search),
        ("Boyer-Moore", boyer_moore_search),
    ]

    for name, func in algorithms:
        matches, elapsed_ms = time_it(func, text, pattern)
        print(f"{name}:")
        print(f"  Matches found at indices: {matches}")
        print(f"  Time taken: {elapsed_ms:.4f} ms\n")

    # Cross-check all three algorithms agree
    results = [func(text, pattern) for _, func in algorithms]
    assert results[0] == results[1] == results[2], "Mismatch between algorithms!"
    print("All three algorithms agree on the match positions. ✔")


if __name__ == "__main__":
    main()

Text length   : 120 characters
Pattern       : 'ABABCABAB' (length 9)

Naive Search:
  Matches found at indices: [10, 64, 94]
  Time taken: 0.0289 ms

Knuth-Morris-Pratt (KMP):
  Matches found at indices: [10, 64, 94]
  Time taken: 0.0254 ms

Boyer-Moore:
  Matches found at indices: [10, 64, 94]
  Time taken: 0.0249 ms

All three algorithms agree on the match positions. ✔
